# Install Requirements

In [1]:
!pip install selenium webdriver-manager dateparser tqdm requests


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Location Discovery

In [ ]:
import csv
import logging
import os
import random
import sys
import threading
import time
from datetime import datetime
from queue import Queue
from typing import List, Tuple

import requests
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.microsoft import EdgeChromiumDriverManager

# CONFIG
CONFIG = {
    "category_url": "https://www.olx.co.id/dijual-rumah-apartemen_c5158",
    "target_provinces": ["DKI Jakarta", "Banten", "Jawa Barat", "Jawa Timur"],
    "nation": "Indonesia",
    "max_retries": 3,
    "wait_seconds": 12,
    "headless": True,
    "respectful_delay_min": 0.8,
    "respectful_delay_max": 2.0,
    "user_agents": [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Edge/120.0.0.0",
    ],
    "robots_root": "https://www.olx.co.id",
}

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s:%(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("olx-location-discovery")

# Globals
WRITE_QUEUE: "Queue[dict]" = Queue()

# Utilities
def respectful_pause():
    time.sleep(random.uniform(CONFIG["respectful_delay_min"], CONFIG["respectful_delay_max"]))

def check_robots(robots_root: str) -> bool:
    try:
        r = requests.get(robots_root.rstrip("/") + "/robots.txt", timeout=6)
        if r.status_code == 200:
            txt = r.text.lower()
            if "disallow" in txt and "/item/" in txt:
                logger.warning("robots.txt mentions /item/ disallow; please inspect manually.")
                return False
        return True
    except Exception as e:
        logger.warning("Could not fetch robots.txt: %s", e)
        return True

def pick_user_agent():
    return random.choice(CONFIG["user_agents"])

def make_driver(user_agent: str = None):
    options = Options()
    if CONFIG["headless"]:
        options.add_argument("--headless=new")
    options.add_argument("--inprivate")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    if user_agent:
        options.add_argument(f"--user-agent={user_agent}")
    driver_path = EdgeChromiumDriverManager().install()
    driver = webdriver.Edge(driver_path, options=options)
    driver.set_page_load_timeout(CONFIG["wait_seconds"] * 2)
    return driver

def safe_get(driver, url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            driver.get(url)
            return True
        except (WebDriverException, TimeoutException) as e:
            wait = (2 ** (attempt - 1)) + random.random()
            logger.warning(f"get({url}) attempt {attempt} failed: {e} — backoff {wait:.1f}s")
            time.sleep(wait)
            continue
    return False

# Anchor finders
def find_location_anchors(driver) -> List[Tuple[str, str]]:
    anchors = []
    try:
        uls = driver.find_elements(By.CSS_SELECTOR, "ul[data-aut-id^='ulLevel_']")
        for ul in uls:
            a_tags = ul.find_elements(By.TAG_NAME, "a")
            for a in a_tags:
                txt = (a.text or "").strip()
                href = a.get_attribute("href")
                if txt and href:
                    anchors.append((txt, href))
        if anchors:
            return anchors
    except Exception:
        pass

    try:
        body_as = driver.find_elements(By.CSS_SELECTOR, "main a, div a, section a")
        for a in body_as:
            txt = (a.text or "").strip()
            href = a.get_attribute("href")
            if txt and href and len(txt) < 80:
                anchors.append((txt, href))
    except Exception:
        pass

    seen = set()
    uniq = []
    for t, h in anchors:
        key = (t.lower(), h)
        if key not in seen:
            seen.add(key)
            uniq.append((t, h))
    return uniq

def find_matching_links(anchors: List[Tuple[str, str]], targets: List[str]) -> List[Tuple[str, str]]:
    matches = []
    targets_lower = [t.lower() for t in targets]
    for txt, href in anchors:
        tl = txt.lower()
        for tgt in targets_lower:
            if tgt in tl:
                matches.append((txt, href))
                break
    return matches

# Extraction functions
def extract_province_links(driver, desired_provinces: List[str]):
    if not safe_get(driver, CONFIG["category_url"], retries=CONFIG["max_retries"]):
        raise RuntimeError("Failed to open category page")
    WebDriverWait(driver, CONFIG["wait_seconds"]).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    respectful_pause()
    anchors = find_location_anchors(driver)
    matches = find_matching_links(anchors, desired_provinces)
    logger.info(f"Found {len(matches)} matching province links")
    return matches

def extract_city_links(driver, province_name, province_url):
    if not safe_get(driver, province_url):
        return []
    respectful_pause()
    anchors = find_location_anchors(driver)
    results = []
    for txt, href in anchors:
        if txt and txt.lower() not in province_name.lower() and len(txt) < 80:
            results.append((txt, href))
    seen = set()
    uniq = []
    for t, h in results:
        k = (t.lower(), h)
        if k not in seen:
            seen.add(k)
            uniq.append((t, h))
    return uniq

def extract_district_links(driver, province, city, city_url):
    if not safe_get(driver, city_url):
        return []
    respectful_pause()
    anchors = find_location_anchors(driver)
    candidates = []
    for txt, href in anchors:
        if txt and len(txt) <= 80:
            candidates.append((txt, href))
    seen = set()
    uniq = []
    for t, h in candidates:
        k = (t.lower(), h)
        if k not in seen:
            seen.add(k)
            uniq.append((t, h))
    return uniq

# CSV writer
def csv_writer_worker(output_path: str, fieldnames: List[str]):
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        while True:
            row = WRITE_QUEUE.get()
            if row is None:
                break
            writer.writerow(row)
            WRITE_QUEUE.task_done()

# Core Discovery Logic
def run_discovery_core(output_path: str, target_provinces: List[str]):
    check_robots(CONFIG["robots_root"])
    driver = make_driver(pick_user_agent())
    try:
        provinces = extract_province_links(driver, target_provinces)
        for prov_text, prov_url in provinces:
            prov_name = next((p for p in target_provinces if p.lower() in prov_text.lower()), prov_text)
            cities = extract_city_links(driver, prov_name, prov_url)
            if not cities:
                cities = [(prov_name, prov_url)]
            for city_text, city_url in cities:
                districts = extract_district_links(driver, prov_name, city_text, city_url)
                if not districts:
                    WRITE_QUEUE.put({
                        "nation": CONFIG["nation"],
                        "province": prov_name,
                        "city": city_text,
                        "district": city_text,
                        "district_url": city_url,
                        "extracted_at": datetime.utcnow().isoformat()
                    })
                else:
                    for dist_text, dist_url in districts:
                        WRITE_QUEUE.put({
                            "nation": CONFIG["nation"],
                            "province": prov_name,
                            "city": city_text,
                            "district": dist_text,
                            "district_url": dist_url,
                            "extracted_at": datetime.utcnow().isoformat()
                        })
                respectful_pause()
    finally:
        driver.quit()

# Notebook helper function
def run_location_discovery(province: str, output_csv: str):
    original_targets = CONFIG["target_provinces"].copy()
    matches = [p for p in original_targets if province.lower() in p.lower()]
    if not matches:
        raise ValueError(f"Province '{province}' not found in target_provinces: {original_targets}")

    fieldnames = ["nation", "province", "city", "district", "district_url", "extracted_at"]
    writer_thread = threading.Thread(target=csv_writer_worker, args=(output_csv, fieldnames), daemon=True)
    writer_thread.start()

    start = time.time()
    try:
        run_discovery_core(output_csv, matches)
    finally:
        WRITE_QUEUE.put(None)
        writer_thread.join()
    logger.info(f"✅ Discovery for {matches[0]} done in {time.time()-start:.1f} s. Saved to {output_csv}")

## Tes Provinsi Jawa Timur

In [ ]:
run_location_discovery(province="Jawa Timur", output_csv="districts_jatim.csv")

## Cek Hasil CSV Jawa Timur

In [ ]:
import pandas as pd

df = pd.read_csv("districts_jatim.csv")
df.head(10)

## Konversi ke Excel

In [ ]:
import pandas as pd

input_csv = "districts_jatim.csv"
output_excel = "districts_jatim.xlsx"

try:
    df = pd.read_csv(input_csv)

    df.to_excel(output_excel, index=False)

    print(f"✅ File berhasil dikonversi ke: {output_excel}")

except Exception as e:
    print(f"❌ Terjadi error: {e}")